<a href="https://colab.research.google.com/github/gflamiano/SJU-CSC-672-ML/blob/main/HW6/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 6 Analysis

#### 1. How many principal components were required to retain 95% of the total variance compared to the original 4,096 dimensions? Based on your reconstruction plots in Task 3, what visual details are lost at lower values of k; e.g., k=10, and at what point does the reconstructed image become visually indistinguishable from the original?

PCA needed 111 components to retain 95% of the variance (fitted on the 320 training images), compared with the original 4,096 dimensions. That is about 37x fewer dimensions, or roughly 2.7% of the original size. Despite this compression, logistic regression reached the same test accuracy (0.9750) on the 111 PCA features as on the raw pixels, so the discarded 5% of variance carried almost no information useful for identifying subjects.

At k = 10 the reconstructions look like blurry, generic faces. Overall head shape, hair outline, and the broad lighting pattern survive, but identity-specific details are gone: the sharpness of the eyes and eyebrows, glasses, mouth shape and expression, and skin texture. Different people start to look alike. The reconstruction error confirms this: across all 400 faces the MSE is 0.00662 at k = 10, falling to 0.00244 at k = 50 and 0.00125 at k = 100. By k = 50 to 100 the faces are clearly recognizable, and the remaining differences are fine edges and texture. The reconstruction becomes visually indistinguishable from the original at about k = 200, where the MSE has dropped to 0.00041 (roughly 3% of the k = 10 error, and an average pixel error of only about 0.02 on a 0–1 intensity scale). At k = 300 the MSE falls further to 0.00011, but the improvement is no longer visible by eye, which is consistent with 111 components already retaining 95% of the variance. Beyond about k = 200, extra components add very little visible information.

#### 2. What do the earliest principal components (the first few "Eigenfaces") physically capture in the images? Why do later principal components appear increasingly like fine-grained pixel noise?

The first eigenfaces capture the largest sources of variation across the dataset, which are broad, low-spatial-frequency patterns: the direction and strength of lighting (one side of the face bright, the other dark), overall brightness of the face versus the background, hair and head outline, and general face shape. These are the changes that affect the most pixels at once, so they explain the most variance. PC 1 alone explains 23.81% of the total variance, and the first two components together explain 37.81%.

PCA forces each new component to be orthogonal to the previous ones and to explain as much of the *remaining* variance as possible. After the global patterns are used up, what remains is small-scale, localized variation: individual edges, fine texture, small expression differences, and idiosyncrasies of particular photos. Each of these explains only a tiny fraction of the variance, and with just 400 samples the later components also fit sample-specific noise. This is why 123 components are needed to reach 95% of the variance and 260 to reach 99%: the last few percent are spread across many components that each contribute very little. Their images therefore look like high-frequency speckle, and they contribute little when reconstructing a face.

#### 3. Why is it critical to fit the PCA transformer strictly on the training split rather than on the entire dataset prior to splitting? What negative impacts occur if PCA is fitted on the full dataset before training downstream classifiers?

PCA learns its components, its mean, and the number of components needed for a variance threshold from whatever data it sees. Our own results show this: fitted on all 400 faces (Section 2), 123 components were needed for 95% of the variance, but fitted on the 320 training faces only (Section 5), 111 were needed. The learned feature space changes with the images included, so if the test images are included when PCA is fitted, information about them (their mean and their variance directions) shapes the space the classifier is trained in. This is data leakage: the test set is no longer truly unseen.

The first negative impact is optimistic accuracy. Because the projection is tuned partly to the test images, test accuracy overestimates how the model would perform on new faces. This also makes model comparisons less trustworthy, since leakage can favor some models over others. Finally, it invalidates the evaluation of the whole pipeline. In deployment, new faces arrive after PCA is fixed, so the evaluation should mimic that by fitting PCA on the training data and only calling `transform` on the test data.

PCA is unsupervised (it never sees labels), so the leakage is milder than if a supervised step were fitted on all the data. It is still a violation of the train/test separation, and good practice is to treat PCA as part of the model. In Sections 2 to 4, PCA was fitted on all faces for exploration and visualization only, which is fine because no test accuracy was reported from it.

#### 4. Compare the classification accuracy of Logistic Regression against Kernel SVM and the Ensemble Classifier on PCA-reduced features. Did non-linear kernels or ensemble methods yield substantial accuracy gains? Explain why classical classifiers (linear, non-linear, or ensemble) hit a performance ceiling when operating directly on flattened pixel vectors or global PCA projections.

On the same PCA-reduced features, Logistic Regression reached a test accuracy of 0.9750 (78 of 80 faces correct), the RBF SVM reached 0.9500 (76 of 80), and the Soft Voting Ensemble reached 0.9750 (78 of 80). Neither the non-linear kernel nor the ensemble improved on the linear baseline: the RBF SVM was two images worse, and the ensemble only matched it. The macro F1 scores tell the same story (0.9733 for logistic regression versus 0.9467 for the SVM), and logistic regression classified both test images correctly for 38 of the 40 subjects, compared with 36 for the SVM. Because the baseline was already at 97.5%, there was almost no room to improve, since the best any model could do was gain two more images. With a test set this small, each image is worth 1.25 points, so differences of one or two images are within noise and should not be read as a real advantage for any model.

The ensemble members show where the ensemble's result came from. The RBF SVM scored 0.9500, the Random Forest 0.9625, and the polynomial SVM only 0.7875. The ensemble scored higher than any of its three members, which suggests that the RBF SVM and Random Forest made different mistakes that the averaged probabilities partly corrected, and the weak polynomial SVM did not drag the result down. This gain only brought the ensemble level with the linear model, and it came at a higher cost: 1.33s of training, against 0.03s for the SVM alone. The polynomial SVM's weak result is likely a matter of its default settings, since a degree-2 kernel with default `C` and `gamma` was not tuned for this data, but that is a hypothesis we did not test. Flexible models also have little data to learn from, with only 8 training images per subject, so after PCA the faces of each subject already form fairly compact clusters that a linear boundary can separate well.

More generally, classical classifiers hit a ceiling because the limit lies in the representation, not in the classifier. They treat a flattened image as a list of independent numbers, so they have no notion that neighboring pixels form edges, eyes, or a nose, and they cannot recognize the same feature when it shifts position. PCA is also unsupervised, so it keeps the directions of greatest variance, which can reflect lighting or background rather than identity. Swapping in a more complex classifier on the same features cannot add information that the features do not contain. Olivetti is a favorable case for these methods, because the faces are aligned, centered, and photographed under controlled conditions, which is why accuracy is already high here.

#### 5. Based on your findings in Task 8 regarding accuracy and training efficiency, why are global linear transformations like PCA and classical pixel classifiers insufficient for complex computer vision tasks?

Task 8 shows that PCA is an effective compression tool but not a way to gain accuracy. Reducing 4,096 pixel features to 111 principal components cut logistic regression's training time from 28.51s to 0.55s (about 52x faster, and the 0.55s even includes fitting PCA), while test accuracy stayed identical at 0.9750. The more elaborate models did not improve on this: the RBF SVM scored 0.9500 and trained in 0.03s on the already-reduced features, and the soft voting ensemble matched the linear baseline at 0.9750 while taking 1.33s, about two and a half times longer than the PCA logistic regression. In other words, the extra model complexity cost more time without raising accuracy, and the only thing that clearly helped efficiency was shrinking the representation. This suggests that the performance limit comes from the features, not the classifier.

Our results should not be read as proof that these methods work in general, because Olivetti is an unusually favorable case. The faces are aligned, centered, similarly scaled, and photographed under controlled lighting against a plain background, so the pixel vector for a given person varies only modestly between images. Under those conditions, a flattened pixel vector and a linear boundary are enough to reach high accuracy. The same pipeline performed very differently on the Oxford 102 Flowers dataset earlier in this module, where logistic regression on raw and PCA-reduced pixels reached only about 12.5 to 13.5% accuracy across 102 classes. That contrast shows where global linear methods break down: when images are unaligned and cluttered, and the object varies in position, scale, pose, and background.

There are several reasons for this failure. Flattening an image into a vector discards its 2D layout, so the model cannot exploit locality, meaning edges, textures, and parts. A flattened pixel vector also has no translation or pose invariance, so a shifted or rotated object produces a very different vector even though the content is the same. PCA keeps the directions of greatest variance, which in cluttered images are often driven by lighting and background rather than by the object's identity, and because it is unsupervised and linear, it cannot build the hierarchy that recognition needs, where edges combine into parts and parts into objects. Finally, the feature count grows with image size and color channels (4,096 here, 12,288 for the 64x64 RGB flower images) while the number of labeled samples stays small. This motivates Convolutional Neural Networks and pre-trained deep feature extractors, which share weights across positions, learn local features and their hierarchy, gain invariance to shifts, and train end-to-end for the classification goal rather than for variance.